# Advertising Benchmark

This notebook checks whether the learned policy spends advertising effort where it actually accelerates adoption.

## Environment Basics

- State space: $\mathcal{X}=\{N,C\}$, where $N$ is noncustomer/uninformed and $C$ is customer/informed.
- Action space: $\mathcal{A}=\{0,1\}$, where action $1$ displays an advertisement.
- Population law: $\mu_t\in\Delta(\{N,C\})$, summarized by $p_t=\mu_t(C)$.
- Goal: increase $p_t$ quickly while paying the advertising cost.
- Reference: finite-horizon dynamic-programming policy/value on the population grid.

The benchmark reward has the form

$$
r_t(x,a,\mu_t)=\mathbf{1}_{\{x=C\}}-c_{ad}a,
\qquad
J(\theta)=\mathbb{E}\left[\sum_{t=0}^{T-1} r_t(X_t,A_t,\mu_t)\right].
$$


In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mfc.experiments import notebook_helpers as nh

ENV_NAME = "advertising"
BASE_DIR = ROOT / "runs" / "extended_benchmark_bundles" / ENV_NAME
PRESET = "mid"
QUICK = PRESET == "smoke"
RUN_MISSING = False
FORCE_REBUILD = False
EXTENDED = True


In [ ]:
bundle = (
    nh.ensure_discrete_benchmark_bundle(
        ENV_NAME,
        BASE_DIR,
        quick=QUICK,
        force=FORCE_REBUILD,
        extended=EXTENDED,
        preset=PRESET,
    )
    if RUN_MISSING
    else nh.bundle_paths(ENV_NAME, BASE_DIR)
)

histories = nh.load_training_histories(bundle)
application = nh.load_application_data(bundle)
studies = nh.load_study_data(bundle)
grid_metrics = nh.load_study_grid_metrics(bundle)
diagnostics = nh.load_diagnostic_data(bundle)
optimization_history = nh.load_optimization_history(bundle)

bundle


## Main Results

The main checks are adoption, spending, and value.

1. **Validation value over training**: $J(\theta_k)$ compared with the DP reference.
2. **Adoption trajectory**: $p_t=\mu_t(C)$ with target-level guides at 50% and 80%.
3. **Advertising rate**: the learned probability/intensity of advertising over time.
4. **Objective components**: cumulative population gain and cumulative advertising cost.


In [ ]:
display(nh.discrete_main_summary_table(ENV_NAME, application))
nh.plot_discrete_main_results(ENV_NAME, histories, application)


## Estimator Diagnostic Appendix

These plots are supporting checks for the estimator chain. They are useful when a training curve looks suspicious, but they are not the first thing to interpret.

The panels summarize

$$
\mathbb{E}[d(M^\lambda,\mu)],
\qquad
\operatorname{MSE}(\widehat g)=\mathbb{E}\|\widehat g-g\|_2^2,
\qquad
\operatorname{tr}\operatorname{Cov}(S_\lambda),
\qquad
\mathbb{E}\|\widehat D_t-D_t\|_2^2.
$$

Here $S_\lambda=\nabla_\theta\log q_\lambda(M)$ is the population-law score and $D_t=\partial\mu_t/\partial\theta$ is the population sensitivity.


In [ ]:
nh.plot_discrete_diagnostic_appendix(diagnostics)


## Total-Variation Calibration

For finite laws $p,q\in\Delta(\mathcal{X})$,

$$
d_{TV}(p,q)=\frac{1}{2}\|p-q\|_1.
$$

The simplex perturbation has a pathwise TV bound proportional to $\lambda$. The logit/CLR perturbation is checked against its reference radius. The table reports empirical expectations, quantiles, and violation rates.


In [ ]:
tv_table = nh.perturbation_tv_comparison_table(diagnostics)
display(tv_table)
nh.plot_perturbation_tv_comparison(diagnostics)


## Optional Extended Studies

Run this section when checking budget allocation, horizon scaling, adaptive lambda, ablations, or robustness. These plots are intentionally separated from the main benchmark result.


In [ ]:
nh.plot_budget_and_horizon(studies)
nh.plot_budget_pareto(studies, grid_metrics)
nh.plot_lambda_training_comparison(studies)
nh.plot_optimization_history(optimization_history)
nh.plot_optimization_summary(studies)
nh.plot_extended_study_summaries(studies, grid_metrics)


## Figure Coverage Audit

These tables map the broader requested figure list to saved artifacts. They are useful for checking completeness; they are not meant to be the headline story of the benchmark.


In [ ]:
nh.figure_checklist(ENV_NAME)


In [ ]:
nh.figure_coverage_matrix(ENV_NAME)


## Raw Tables For Custom Figures

The first few rows below are the main saved tables used by the plots above. Use these as the entry point for custom figures without rerunning training.


In [ ]:
sample_algorithm = next(iter(application))
sample_diag_algorithm = next(iter(diagnostics))
(
    application[sample_algorithm]["time_metrics"].head(),
    diagnostics[sample_diag_algorithm].get("gradient", pd.DataFrame()).head(),
    studies.get("budget", pd.DataFrame()).head(),
)
